# M4-B2 — Vision PCB Defect (binôme async)

Auteurs : `<prénom1>` × `<prénom2>` — Date : `<date>`

**Conventions** :
- `random_state=42`
- Pas de `print` excessif
- `Co-authored-by:` sur les commits significatifs

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

sys.path.append('..')  # racine du repo → permet les imports « package » src.*
from src.load_data import CLASSES, get_dataloaders

DATA_DIR = Path('../data/pcb_defect_sample')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")

Device : cpu


## 1. EDA dataset PCB (~1h binôme)

- Distribution des 7 classes (déséquilibre ?)
- Visualisation 7×3 subplot (3 exemples par classe)
- Notes : qualité d'image, variabilité intra-classe, ambiguïtés

In [ ]:

# Distribution des classes
from collections import Counter
from src.load_data import PCBDefectDataset, CLASSES

dataset_full = PCBDefectDataset(DATA_DIR)
counts = Counter(CLASSES[label] for _, label in dataset_full.samples)

# Affichage texte
print("Nombre total d'images :", len(dataset_full))
print("\nDistribution par classe :")
for cls in CLASSES:
    n = counts[cls]
    bar = "█" * (n // 10)
    print(f"  {cls:<12} {n:>4}  {bar}")

# Graphique
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(CLASSES, [counts[c] for c in CLASSES], color="steelblue", edgecolor="white")
ax.set_title("Distribution des classes — PCB Defect Sample")
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'images")
ax.tick_params(axis='x', rotation=15)
for i, cls in enumerate(CLASSES):
    ax.text(i, counts[cls] + 2, str(counts[cls]), ha='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:

# 3 exemples par classe — grille 7 lignes × 3 colonnes
import random

random.seed(42)

fig, axes = plt.subplots(7, 3, figsize=(7, 16))
fig.suptitle("3 exemples par classe — PCB Defect Sample", fontsize=13, y=1.01)

# Regrouper les chemins par classe
from collections import defaultdict
samples_by_class = defaultdict(list)
for path, label in dataset_full.samples:
    samples_by_class[CLASSES[label]].append(path)

for row, cls in enumerate(CLASSES):
    paths = samples_by_class[cls]
    chosen = random.sample(paths, min(3, len(paths)))
    for col, img_path in enumerate(chosen):
        ax = axes[row][col]
        img = Image.open(img_path).convert("L")
        ax.imshow(img, cmap="gray")
        ax.axis("off")
        if col == 0:
            ax.set_title(cls, fontsize=10, loc="left", pad=4)
    # cases vides si moins de 3 images
    for col in range(len(chosen), 3):
        axes[row][col].axis("off")

plt.tight_layout()
plt.show()



### Observations EDA

| Axe | Constat |
|-----|---------|
| **Équilibre** | ≈ 300 images / classe → dataset équilibré, pas besoin de pondération |
| **Résolution** | 64×64 px niveaux de gris — faible résolution, peu de texture fine |
| **Variabilité intra-classe** | `ok` très homogène ; `copper` et `spur` montrent des formes très diverses |
| **Ambiguïtés** | `open` vs `mousebite` parfois difficiles à distinguer à l'œil nu |
| **Qualité** | Contraste bon ; quelques images très sombres dans `pin_hole` |


## 2. Implémentation de l'option choisie (~4h binôme)

Cf. `decisions.md` pour le choix (A / B / C).

- Option A : `src/option_a_cnn.py`
- Option B : `src/option_b_transfer.py`
- Option C : `src/option_c_clip.py`

In [ ]:
# TODO — entraîner ou inférer l'option choisie
# Import : from src.option_a_cnn import SimpleCNN, train_one_epoch, evaluate  (idem b/c)
# Mesure : temps train (si options A ou B), latence inférence (toutes options), accuracy

## 3. Comparaison économique 3 approches (~1h30)

Voir `economic_comparison.md` à remplir.

## 4. Verdict + préparation restitution duo (~1h30)

- `verdict.md` : recommandation, 8 lignes max
- Préparation restitution mardi 1ᵉʳ sept (rentrée M5) : qui dit quoi ?